## Cell 1

In [17]:
"""Phase 2.1 — Fine-tune DistilBERT on scikit-learn issue classification.

This notebook trains the encoder, evaluates on val during training, then
evaluates ONCE on test and writes the model card. The resulting artifacts
(classifier.pt, tokenizer/, model_card.json) are saved to
backend/data/classifier_artifacts/ and pushed to MinIO in Sitting B.

Inspired by the instructor's chapter-3 NLP and chapter-7 fine-tuning notebooks.
"""
import hashlib
import json
import os
import random
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import torch
import wandb
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Instructor's W&B service-wait pattern — handles slow networks during init.
os.environ["WANDB__SERVICE_WAIT"] = "300"

# Reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Cell 2

In [ ]:
# Class distribution in train (for class weights below)
from collections import Counter

DATA_DIR = Path("..") / "data" / "issues" / "splits"
LABELS = ["bug", "feature", "docs", "question"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}


def load_jsonl(path: Path) -> list[dict]:
    with path.open() as f:
        return [json.loads(line) for line in f]


def issue_to_text(row: dict) -> str:
    """Concatenate title and body into one text field for the encoder."""
    title = (row.get("title") or "").strip()
    body = (row.get("body") or "").strip()
    return f"{title}\n\n{body}" if body else title


train_rows = load_jsonl(DATA_DIR / "train.jsonl")
val_rows = load_jsonl(DATA_DIR / "val.jsonl")
test_rows = load_jsonl(DATA_DIR / "test.jsonl")

print(f"train: {len(train_rows)}")
print(f"val:   {len(val_rows)}")
print(f"test:  {len(test_rows)}")

train_dist = Counter(r["label"] for r in train_rows)
print(f"\ntrain distribution: {dict(train_dist)}")

train: 2690
val:   576
test:  578

train distribution: {'feature': 798, 'docs': 673, 'bug': 1023, 'question': 196}


## Cell 3

In [19]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256  # most issue titles+bodies fit; the rest are truncated.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def rows_to_dataset(rows: list[dict]) -> Dataset:
    return Dataset.from_dict(
        {
            "text": [issue_to_text(r) for r in rows],
            "label": [LABEL_TO_ID[r["label"]] for r in rows],
        }
    )


train_ds = rows_to_dataset(train_rows)
val_ds = rows_to_dataset(val_rows)
test_ds = rows_to_dataset(test_rows)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # the data collator handles padding per-batch
    )


train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

print("tokenized.")

Map: 100%|██████████| 578/578 [00:00<00:00, 1908.03 examples/s]

tokenized.


## Cell 4

In [20]:
# Class weights: balanced formula = n_total / (n_classes * n_class_i)
# Smaller classes get higher weight in the loss.
total = sum(train_dist.values())
n_classes = len(LABELS)
class_weights = torch.tensor(
    [total / (n_classes * train_dist[label]) for label in LABELS],
    dtype=torch.float,
    device=device,
)
print(f"class weights (bug, feature, docs, question): {class_weights.tolist()}")

# The model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_classes,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)


# Custom Trainer subclass that applies class weights to the loss.
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


# W&B project
wandb.init(
    project="maintainer-copilot",
    name=f"distilbert-classifier-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}",
    config={
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "labels": LABELS,
        "train_size": len(train_rows),
        "val_size": len(val_rows),
        "test_size": len(test_rows),
    },
)

# Training arguments
OUTPUT_DIR = Path("..") / "data" / "classifier_artifacts" / "checkpoints"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    fp16=torch.cuda.is_available(),  # mixed precision when GPU available
    seed=SEED,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

print("trainer ready.")

class weights (bug, feature, docs, question): [0.6573802828788757, 0.8427318334579468, 0.9992570877075195, 3.4311225414276123]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11044.91it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainer ready.


## Cell 5

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.973933,0.914271,0.755208,0.663097
2,0.678990,1.088737,0.750000,0.631621


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]


TrainOutput(global_step=338, training_loss=0.9285532990856283, metrics={'train_runtime': 43.3701, 'train_samples_per_second': 248.097, 'train_steps_per_second': 15.587, 'total_flos': 356350011924480.0, 'train_loss': 0.9285532990856283, 'epoch': 2.0})

## Cell 6

In [22]:
# This is the only time we touch the test set.
test_predictions = trainer.predict(test_ds)
test_logits = test_predictions.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_labels = test_predictions.label_ids

# Top-line metrics
test_accuracy = accuracy_score(test_labels, test_preds)
test_macro_f1 = f1_score(test_labels, test_preds, average="macro")

# Per-class F1
per_class_report = classification_report(
    test_labels,
    test_preds,
    target_names=LABELS,
    output_dict=True,
    zero_division=0,
)
per_class_f1 = {label: per_class_report[label]["f1-score"] for label in LABELS}

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds, labels=list(range(n_classes)))

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro-F1: {test_macro_f1:.4f}")
print("\nPer-class F1:")
for label, f1 in per_class_f1.items():
    print(f"  {label:10s}: {f1:.4f}")
print(f"\nConfusion matrix (rows=true, cols=pred), order: {LABELS}")
print(cm)

# Log final test metrics to W&B
wandb.log(
    {
        "test/accuracy": test_accuracy,
        "test/macro_f1": test_macro_f1,
        **{f"test/f1_{label}": per_class_f1[label] for label in LABELS},
    }
)

Test accuracy: 0.8478
Test macro-F1: 0.7462

Per-class F1:
  bug       : 0.9255
  feature   : 0.8148
  docs      : 0.8845
  question  : 0.3600

Confusion matrix (rows=true, cols=pred), order: ['bug', 'feature', 'docs', 'question']
[[261   1   3   9]
 [  4  77   2   6]
 [ 10   4 134   1]
 [ 15  18  15  18]]


## Cell 7

In [ ]:
assert wandb.run is not None, "wandb.run is None — cell 4 must run before cell 7"

ARTIFACT_DIR = Path("..") / "data" / "classifier_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save the model state dict — smaller than the full model dir.
classifier_path = ARTIFACT_DIR / "classifier.pt"
torch.save(model.state_dict(), classifier_path)
print(f"saved {classifier_path}")

# Save tokenizer alongside (its config is needed at inference time).
tokenizer_dir = ARTIFACT_DIR / "tokenizer"
tokenizer.save_pretrained(tokenizer_dir)
print(f"saved {tokenizer_dir}/")


# Compute SHA-256 of the .pt file.
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


classifier_sha = sha256_of(classifier_path)
print(f"classifier.pt SHA-256: {classifier_sha}")

# Training data hash from D-008
training_data_hash = "1a4e887a580b5289d4b87fcff2890235c95945d78cd768f3e25933b3ca4c3959"

# Model card — committed schema in app/db/models, ARCH.md §10
model_card = {
    "sha256": classifier_sha,
    "backbone": MODEL_NAME,
    "tokenizer": MODEL_NAME,
    "num_labels": n_classes,
    "labels": LABELS,
    "freeze_policy": "full_finetune",  # entire DistilBERT is trained
    "hyperparameters": {
        "max_length": MAX_LENGTH,
        "epochs": training_args.num_train_epochs,
        "batch_size_train": training_args.per_device_train_batch_size,
        "batch_size_eval": training_args.per_device_eval_batch_size,
        "learning_rate": training_args.learning_rate,
        "weight_decay": training_args.weight_decay,
        "warmup_ratio": training_args.warmup_ratio,
        "class_weights": class_weights.tolist(),
        "seed": SEED,
        "fp16": training_args.fp16,
    },
    "training_data_sha256": training_data_hash,
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "per_class_f1": per_class_f1,
    "confusion_matrix": cm.tolist(),
    "trained_at": datetime.now(UTC).isoformat(),
    "wandb_run": {
        "project": wandb.run.project,
        "id": wandb.run.id,
        "url": wandb.run.url,
    },
    "env_fingerprint": {
        "torch": torch.__version__,
        "cuda": torch.version.cuda if torch.cuda.is_available() else None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "transformers": __import__("transformers").__version__,
    },
}

card_path = ARTIFACT_DIR / "model_card.json"
with card_path.open("w") as f:
    json.dump(model_card, f, indent=2)
print(f"saved {card_path}")

# Done.
wandb.finish()
print("\ndone.")

saved ../data/classifier_artifacts/classifier.pt
saved ../data/classifier_artifacts/tokenizer/
classifier.pt SHA-256: a3bd4cb8f9328ce409169d14ef4585c27f1149ff2c69795de0e8e5759a8f3a59
saved ../data/classifier_artifacts/model_card.json


eval/accuracy,█▁
eval/loss,▁█
eval/macro_f1,█▁
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
test/accuracy,▁▁
test/f1_bug,▁
test/f1_docs,▁
test/f1_feature,▁
+11,...



done.


## Cell 8

In [ ]:
# Sanity: load the saved artifacts in a fresh model object and predict on one example.
fresh_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_classes,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)
fresh_model.load_state_dict(torch.load(classifier_path, map_location=device))
fresh_model.to(device)
fresh_model.eval()

fresh_tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)

sample_issue = "Random Forest fit fails with sparse input"
tokens = fresh_tokenizer(
    sample_issue, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
).to(device)
with torch.no_grad():
    logits = fresh_model(**tokens).logits
    probs = torch.softmax(logits, dim=-1)[0]

predicted = ID_TO_LABEL[probs.argmax().item()]
print(f"Sample: {sample_issue!r}")
print(f"Predicted: {predicted}")
print(f"Confidence: {probs.max().item():.4f}")
print("\nAll classes:")
for label, p in sorted(zip(LABELS, probs.tolist(), strict=True), key=lambda x: -x[1]):
    print(f"  {label:10s}: {p:.4f}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16755.10it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sample: 'Random Forest fit fails with sparse input'
Predicted: feature
Confidence: 0.4321

All classes:
  feature   : 0.4321
  bug       : 0.2406
  question  : 0.2379
  docs      : 0.0894
